# Phase 6: thresholds and business costs, on half A

**CRISP-DM stage:** Evaluation

**Runs on:** the laptop, from the scores saved by the Phase 5 Kaggle runs. No photos are read and no model runs.

**These are development numbers on half A.** Lines chosen here are applied unchanged to half B only in the Final test, after every choice is locked.

**The business assumptions:** a missed defect costs 10 times a false reject, tested from 2 to 100. Defects happen in 1 percent of parts on the production line, tested at 0.1, 1 and 10 percent. Both are stated assumptions, because real costs and rates are not available for a benchmark.

**An honest limit:** a cost-based line is chosen on half A and measured on the same half A photos here, so its cost is optimistic. The fair number comes from half B in the Final test.

## 1. Load the saved scores

For each configuration, product and seed: the score of every half A photo, whether it is defective, and the benchmark threshold. PaDiM has one seed and PatchCore three.

In [ ]:
import csv
import json
import statistics
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import torch

# Lets the notebook run from the project folder or from the notebooks folder
PROJECT_FOLDER = Path.cwd()
if PROJECT_FOLDER.name == "notebooks":
    PROJECT_FOLDER = PROJECT_FOLDER.parent
# Python looks for src in the project folder
sys.path.insert(0, str(PROJECT_FOLDER))
from src import cost

CORE_FOLDER = PROJECT_FOLDER / "outputs" / "results" / "05_core"
RESULTS_FOLDER = PROJECT_FOLDER / "outputs" / "results" / "06_evaluation"
FIGURES_FOLDER = PROJECT_FOLDER / "outputs" / "figures"
RESULTS_FOLDER.mkdir(parents=True, exist_ok=True)

CONFIGS = ["C1", "C2", "C4", "C5"]
CONFIG_NAMES = {"C1": "C1 PaDiM, ResNet-18", "C2": "C2 PaDiM, Wide ResNet-50",
                "C4": "C4 PatchCore, ResNet-18", "C5": "C5 PatchCore, Wide ResNet-50"}
PRODUCTS = ["sheet_metal", "vial"]
PRODUCT_NAMES = {"sheet_metal": "Sheet Metal", "vial": "Vial"}

runs = []
for config in CONFIGS:
    for product in PRODUCTS:
        for metric_file in sorted((CORE_FOLDER / config).glob(config + "_" + product + "_seed*_metrics.json")):
            with open(metric_file) as file:
                metrics = json.load(file)
            scores = []
            labels = []
            with open(str(metric_file).replace("_metrics.json", "_photo_scores.csv"), newline="") as file:
                for photo in csv.DictReader(file):
                    scores.append(float(photo["score"]))
                    labels.append(photo["label"] == "bad")
            runs.append({"config": config, "product": product, "seed": metrics["seed"], "metrics": metrics,
                         "scores": torch.tensor(scores), "labels": torch.tensor(labels)})
print("Runs loaded:", len(runs))

## 2. The benchmark threshold and its weaknesses

The benchmark sets one line at the mean plus 3 standard deviations of every pixel value in good validation photos. A photo is rejected if any pixel is above it.

**Its weakness:** the line comes from single pixels, but a photo is judged by its most suspicious pixel. A photo has about 65,000 pixels, so even a good photo often has one pixel above a line that only 1 pixel in a few hundred crosses. Where the line lands also depends on how each configuration's scores happen to spread, not on the factory's costs. The table shows how differently it behaves.

In [ ]:
def mean_over_seeds(config, product, name):
    """Return the mean of one saved measure over a configuration's seeds."""
    values = []
    for run in runs:
        if run["config"] == config and run["product"] == product:
            values.append(float(run["metrics"][name]))
    return statistics.mean(values)


bench_rows = []
for product in PRODUCTS:
    for config in CONFIGS:
        bench_rows.append({
            "product": product,
            "config": config,
            "image_f1": mean_over_seeds(config, product, "image_f1"),
            "flag_everything_image_f1": mean_over_seeds(config, product, "flag_everything_image_f1"),
            "pixel_f1": mean_over_seeds(config, product, "pixel_f1"),
            "good_validation_flagged": mean_over_seeds(config, product, "good_validation_photos_flagged"),
            "good_validation_photos": mean_over_seeds(config, product, "good_validation_photos"),
            "good_half_a_flagged": mean_over_seeds(config, product, "good_half_a_photos_flagged"),
            "defective_half_a_flagged": mean_over_seeds(config, product, "defective_half_a_photos_flagged"),
        })

with open(RESULTS_FOLDER / "benchmark_threshold_table.csv", "w", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=list(bench_rows[0].keys()))
    writer.writeheader()
    for row in bench_rows:
        writer.writerow(row)

for row in bench_rows:
    print(PRODUCT_NAMES[row["product"]], "|", CONFIG_NAMES[row["config"]],
          "| image F1", round(row["image_f1"], 3), "(flag everything", round(row["flag_everything_image_f1"], 3), ")",
          "| pixel F1", round(row["pixel_f1"], 3),
          "| good validation flagged", round(row["good_validation_flagged"], 1), "of", int(row["good_validation_photos"]),
          "| good half A flagged", round(row["good_half_a_flagged"], 1),
          "| defective half A flagged", round(row["defective_half_a_flagged"], 1))

## 3. The cost-based threshold

For each configuration, product, seed, cost ratio and defect rate, every possible line is tried, from rejecting everything to passing everything, and the cheapest is kept. The cost of the benchmark line is worked out in the same way, so the two can be compared. All sums come from `src/cost.py`.

- **Passing everything** costs the defect rate times the cost ratio: 0.1 per part at 1 percent and ratio 10.
- **Rejecting everything** costs the share of good parts: 0.99 per part at 1 percent.
- A line is only worth using if it beats passing everything.

In [ ]:
cost_rows = []
for run in runs:
    top_score = float(run["scores"].max())
    for defect_rate in cost.DEFECT_RATES:
        for cost_ratio in cost.COST_RATIOS:
            best = cost.cheapest_threshold(run["scores"], run["labels"], defect_rate, cost_ratio)
            benchmark = cost.cost_at(run["scores"], run["labels"], run["metrics"]["threshold"], defect_rate, cost_ratio)
            cost_rows.append({
                "config": run["config"], "product": run["product"], "seed": run["seed"],
                "defect_rate": defect_rate, "cost_ratio": cost_ratio,
                "cheapest_threshold": best["threshold"], "cheapest_cost": best["cost"],
                "cheapest_miss_rate": best["miss_rate"], "cheapest_false_reject_rate": best["false_reject_rate"],
                # A line above every score passes every photo
                "cheapest_is_pass_everything": best["threshold"] > top_score,
                "benchmark_threshold": run["metrics"]["threshold"], "benchmark_cost": benchmark["cost"],
                "benchmark_miss_rate": benchmark["miss_rate"], "benchmark_false_reject_rate": benchmark["false_reject_rate"],
                "pass_everything_cost": defect_rate * cost_ratio,
                "reject_everything_cost": 1 - defect_rate,
            })

with open(RESULTS_FOLDER / "cost_by_run.csv", "w", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=list(cost_rows[0].keys()))
    writer.writeheader()
    for row in cost_rows:
        writer.writerow(row)


def seed_mean(config, product, defect_rate, cost_ratio, name):
    """Return the mean of one cost measure over a configuration's seeds."""
    values = []
    for row in cost_rows:
        if (row["config"] == config and row["product"] == product
                and row["defect_rate"] == defect_rate and row["cost_ratio"] == cost_ratio):
            values.append(float(row[name]))
    return statistics.mean(values)


summary_rows = []
for product in PRODUCTS:
    for defect_rate in cost.DEFECT_RATES:
        for cost_ratio in cost.COST_RATIOS:
            for config in CONFIGS:
                summary_rows.append({
                    "product": product, "defect_rate": defect_rate, "cost_ratio": cost_ratio, "config": config,
                    "cheapest_cost": seed_mean(config, product, defect_rate, cost_ratio, "cheapest_cost"),
                    "share_of_seeds_passing_everything": seed_mean(config, product, defect_rate, cost_ratio, "cheapest_is_pass_everything"),
                    "benchmark_cost": seed_mean(config, product, defect_rate, cost_ratio, "benchmark_cost"),
                    "pass_everything_cost": defect_rate * cost_ratio,
                })

with open(RESULTS_FOLDER / "cost_summary.csv", "w", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=list(summary_rows[0].keys()))
    writer.writeheader()
    for row in summary_rows:
        writer.writerow(row)

print("Main assumptions: defect rate 1 percent, cost ratio 10. Cost per part, mean over seeds")
for row in summary_rows:
    if row["defect_rate"] == cost.MAIN_DEFECT_RATE and row["cost_ratio"] == cost.MAIN_COST_RATIO:
        beats_benchmark = row["cheapest_cost"] < row["benchmark_cost"]
        print(PRODUCT_NAMES[row["product"]], "|", CONFIG_NAMES[row["config"]],
              "| cheapest line", round(row["cheapest_cost"], 4),
              "| seeds where it means passing everything", round(row["share_of_seeds_passing_everything"], 2),
              "| benchmark line", round(row["benchmark_cost"], 4),
              "| pass everything", round(row["pass_everything_cost"], 4),
              "| cheaper than benchmark:", beats_benchmark)

## 4. Which configuration is cheapest, and when does that change?

For every product, defect rate and cost ratio, the configuration with the lowest mean cost is listed. If the cheapest line for every configuration is simply passing everything, they all cost the same, and that is shown as a tie. These are half A costs and are optimistic.

In [ ]:
best_rows = []
for product in PRODUCTS:
    for defect_rate in cost.DEFECT_RATES:
        for cost_ratio in cost.COST_RATIOS:
            candidates = []
            for row in summary_rows:
                if row["product"] == product and row["defect_rate"] == defect_rate and row["cost_ratio"] == cost_ratio:
                    candidates.append(row)
            lowest = None
            for row in candidates:
                if lowest is None or row["cheapest_cost"] < lowest["cheapest_cost"]:
                    lowest = row
            tied = []
            for row in candidates:
                # Costs this close count as a tie, as they differ only by rounding
                if abs(row["cheapest_cost"] - lowest["cheapest_cost"]) < 1e-9:
                    tied.append(row["config"])
            best_rows.append({"product": product, "defect_rate": defect_rate, "cost_ratio": cost_ratio,
                              "cheapest_configs": " ".join(tied), "cost": lowest["cheapest_cost"],
                              "pass_everything_cost": defect_rate * cost_ratio,
                              "all_pass_everything": lowest["share_of_seeds_passing_everything"] == 1 and len(tied) == len(CONFIGS)})

with open(RESULTS_FOLDER / "cheapest_by_setting.csv", "w", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=list(best_rows[0].keys()))
    writer.writeheader()
    for row in best_rows:
        writer.writerow(row)

for row in best_rows:
    print(PRODUCT_NAMES[row["product"]], "| defect rate", row["defect_rate"], "| ratio", row["cost_ratio"],
          "| cheapest:", row["cheapest_configs"], "| cost", round(row["cost"], 4),
          "| pass everything", round(row["pass_everything_cost"], 4))

## 5. Figures

All show numbers only, with no photos, so they can be committed. PatchCore panels use seed 42.

- **Score histograms:** good and defective photo scores, with the benchmark line (dashed) and the cheapest line at ratio 10 and 1 percent (solid). A cheapest line above every score means passing everything.
- **Cost curves:** the cost of every line at ratio 10 and 1 percent, with the cheapest marked. Dotted lines show passing and rejecting everything.
- **Confusion matrices** at the cheapest line: correct passes, false rejects, missed defects and correct rejects.
- **Cost across the ranges:** each configuration's cheapest cost for every cost ratio, one panel per product and defect rate, with passing everything as a reference.

In [ ]:
COLOURS = {"C1": "#2a78d6", "C2": "#eb6834", "C4": "#1baf7a", "C5": "#eda100"}
GOOD_COLOUR = "#2a78d6"
DEFECTIVE_COLOUR = "#eb6834"
INK = "#0b0b0b"
MUTED = "#52514e"
FIGURE_DPI = 150


def seed42_run(config, product):
    """Return the run with seed 42 for a configuration and product."""
    for run in runs:
        if run["config"] == config and run["product"] == product and run["seed"] == 42:
            return run


def tidy(axis):
    """Give a panel the same plain look as the other charts."""
    axis.spines["top"].set_visible(False)
    axis.spines["right"].set_visible(False)
    axis.grid(axis="y", color="#e4e4e0", linewidth=0.8)
    axis.set_axisbelow(True)


# Score histograms
figure, axes = plt.subplots(2, 4, figsize=(18, 7.5))
for row_number in range(2):
    for column_number in range(4):
        product = PRODUCTS[row_number]
        config = CONFIGS[column_number]
        run = seed42_run(config, product)
        axis = axes[row_number][column_number]
        good_scores = run["scores"][~run["labels"]].tolist()
        defective_scores = run["scores"][run["labels"]].tolist()
        axis.hist(good_scores, bins=15, color=GOOD_COLOUR, alpha=0.7, label="Good photos", edgecolor="white")
        axis.hist(defective_scores, bins=15, color=DEFECTIVE_COLOUR, alpha=0.7, label="Defective photos", edgecolor="white")
        best = cost.cheapest_threshold(run["scores"], run["labels"], cost.MAIN_DEFECT_RATE, cost.MAIN_COST_RATIO)
        axis.axvline(run["metrics"]["threshold"], color=INK, linestyle="--", linewidth=1.2, label="Benchmark line")
        axis.axvline(best["threshold"], color=INK, linestyle="-", linewidth=1.2, label="Cheapest line, ratio 10, 1%")
        axis.set_title(PRODUCT_NAMES[product] + ", " + CONFIG_NAMES[config], fontsize=9, loc="left")
        axis.set_xlabel("Photo score", fontsize=8)
        tidy(axis)
axes[0][0].set_ylabel("Photos")
axes[1][0].set_ylabel("Photos")
handles, names = axes[0][0].get_legend_handles_labels()
figure.legend(handles, names, frameon=False, fontsize=9, loc="lower center", ncol=4, bbox_to_anchor=(0.5, -0.03))
figure.suptitle("Half A photo scores with both lines (development numbers). PatchCore: seed 42", fontsize=10)
figure.tight_layout()
figure.savefig(FIGURES_FOLDER / "phase6_score_histograms.png", dpi=FIGURE_DPI, bbox_inches="tight")
plt.close(figure)

# Cost curves
figure, axes = plt.subplots(2, 4, figsize=(18, 7.5))
for row_number in range(2):
    for column_number in range(4):
        product = PRODUCTS[row_number]
        config = CONFIGS[column_number]
        run = seed42_run(config, product)
        axis = axes[row_number][column_number]
        lines = cost.candidate_thresholds(run["scores"])
        costs = []
        for line in lines:
            costs.append(cost.cost_at(run["scores"], run["labels"], line, cost.MAIN_DEFECT_RATE, cost.MAIN_COST_RATIO)["cost"])
        best = cost.cheapest_threshold(run["scores"], run["labels"], cost.MAIN_DEFECT_RATE, cost.MAIN_COST_RATIO)
        axis.plot(lines, costs, color=COLOURS[config], linewidth=2)
        axis.plot([best["threshold"]], [best["cost"]], marker="o", markersize=8, color=INK)
        axis.axhline(cost.MAIN_DEFECT_RATE * cost.MAIN_COST_RATIO, color=MUTED, linestyle=":", linewidth=1)
        axis.axhline(1 - cost.MAIN_DEFECT_RATE, color=MUTED, linestyle=":", linewidth=1)
        axis.text(lines[0], cost.MAIN_DEFECT_RATE * cost.MAIN_COST_RATIO + 0.02, "pass everything", fontsize=7, color=MUTED)
        axis.text(lines[0], 1 - cost.MAIN_DEFECT_RATE + 0.02, "reject everything", fontsize=7, color=MUTED)
        axis.set_ylim(0, 1.1)
        axis.set_title(PRODUCT_NAMES[product] + ", " + CONFIG_NAMES[config], fontsize=9, loc="left")
        axis.set_xlabel("Line position (photo score)", fontsize=8)
        tidy(axis)
axes[0][0].set_ylabel("Cost per part")
axes[1][0].set_ylabel("Cost per part")
figure.suptitle("Expected cost of every line, ratio 10, defect rate 1 percent, half A. Dot: cheapest. PatchCore: seed 42", fontsize=10)
figure.tight_layout()
figure.savefig(FIGURES_FOLDER / "phase6_cost_curves.png", dpi=FIGURE_DPI, bbox_inches="tight")
plt.close(figure)

# Confusion matrices at the cheapest line
figure, axes = plt.subplots(2, 4, figsize=(16, 7.5))
for row_number in range(2):
    for column_number in range(4):
        product = PRODUCTS[row_number]
        config = CONFIGS[column_number]
        run = seed42_run(config, product)
        axis = axes[row_number][column_number]
        best = cost.cheapest_threshold(run["scores"], run["labels"], cost.MAIN_DEFECT_RATE, cost.MAIN_COST_RATIO)
        counts = cost.confusion_counts(run["scores"], run["labels"], best["threshold"])
        grid = [[counts["correct_passes"], counts["false_rejects"]], [counts["missed_defects"], counts["correct_rejects"]]]
        axis.imshow(grid, cmap="Blues", vmin=0, vmax=len(run["scores"]))
        cell_names = [["correct passes", "false rejects"], ["missed defects", "correct rejects"]]
        for grid_row in range(2):
            for grid_column in range(2):
                axis.text(grid_column, grid_row, str(grid[grid_row][grid_column]) + "\n" + cell_names[grid_row][grid_column],
                          ha="center", va="center", fontsize=9, color=INK)
        axis.set_xticks([0, 1])
        axis.set_xticklabels(["passed", "rejected"])
        axis.set_yticks([0, 1])
        axis.set_yticklabels(["good", "defective"])
        axis.set_title(PRODUCT_NAMES[product] + ", " + CONFIG_NAMES[config], fontsize=9, loc="left")
figure.suptitle("Confusion matrices at the cheapest line, ratio 10, defect rate 1 percent, half A. PatchCore: seed 42", fontsize=10)
figure.tight_layout()
figure.savefig(FIGURES_FOLDER / "phase6_confusion_matrices.png", dpi=FIGURE_DPI, bbox_inches="tight")
plt.close(figure)

# Cheapest cost across the ranges
figure, axes = plt.subplots(2, 3, figsize=(16, 8), sharex=True)
ratio_positions = list(range(len(cost.COST_RATIOS)))
for row_number in range(2):
    for column_number in range(3):
        product = PRODUCTS[row_number]
        defect_rate = cost.DEFECT_RATES[column_number]
        axis = axes[row_number][column_number]
        for config in CONFIGS:
            values = []
            for cost_ratio in cost.COST_RATIOS:
                values.append(seed_mean(config, product, defect_rate, cost_ratio, "cheapest_cost"))
            axis.plot(ratio_positions, values, color=COLOURS[config], linewidth=2, marker="o", markersize=5, label=CONFIG_NAMES[config])
        pass_values = []
        for cost_ratio in cost.COST_RATIOS:
            pass_values.append(defect_rate * cost_ratio)
        axis.plot(ratio_positions, pass_values, color=MUTED, linestyle=":", linewidth=1.5, label="Pass everything")
        axis.set_xticks(ratio_positions)
        axis.set_xticklabels([str(ratio) for ratio in cost.COST_RATIOS])
        axis.set_title(PRODUCT_NAMES[product] + ", defect rate " + str(round(defect_rate * 100, 1)) + "%", fontsize=9, loc="left")
        axis.set_xlabel("Cost ratio (missed defect / false reject)", fontsize=8)
        tidy(axis)
axes[0][0].set_ylabel("Cheapest cost per part")
axes[1][0].set_ylabel("Cheapest cost per part")
handles, names = axes[0][0].get_legend_handles_labels()
figure.legend(handles, names, frameon=False, fontsize=9, loc="lower center", ncol=5, bbox_to_anchor=(0.5, -0.03))
figure.suptitle("Cheapest cost per part across cost ratios and defect rates, half A, mean over seeds (optimistic: line chosen and measured on half A)", fontsize=10)
figure.tight_layout()
figure.savefig(FIGURES_FOLDER / "phase6_cost_across_ranges.png", dpi=FIGURE_DPI, bbox_inches="tight")
plt.close(figure)
print("Saved four figures")